# SwinV2 Image Classification — DIMER E2E tutorial

**Profile:** `E2E` · **Notebook spec:** 1.0
**Pipeline:** [`kurtvalcorza/swin-classification-pipeline`](https://github.com/kurtvalcorza/swin-classification-pipeline)
**Capability:** image-folder validation → supervised SwinV2 fine-tuning → content-addressed artifact publication → fresh-boundary reload → new-image inference.

## What this notebook does

DIMER's Swin classification pipeline demonstrates 100% standalone, in-kernel execution:

| Layer | What it supplies | Where it comes from |
|---|---|---|
| Upstream model | SwinV2 Tiny/Small weights pretrained on ImageNet-1k (Microsoft Swin Transformer V2, MIT-licensed weights via `timm`) | Hugging Face Hub, pinned to an immutable revision and SHA-256 in the base-model catalog |
| In-Notebook Validator | Freezes dataset logical identity: sample IDs, class labels, `train/`/`val/` ownership, per-file digests | In-kernel standalone implementation complying with DIMER handoff contract |
| In-Kernel Fine-Tuning | Supervised **gradient fine-tuning** of the network (AdamW, cross-entropy), artifact publication, fresh reload and evaluation | 100% in-kernel execution (no external worker repository clones or CLI subprocesses) |
| This notebook | Orchestration, tutorial sample/BYOD, baseline, fresh-boundary verification, machine-readable exports | You are here |

**Adaptation type:** full-network gradient fine-tuning. This is not zero-shot inference, in-context conditioning, or preprocessing-only fitting. At inference time the fine-tuned network maps a 256×256 RGB image to one logit per class; the decision rule is `argmax`.

## By the end of this notebook you will be able to

- bootstrap the pipeline at the **immutable revision the release pins**;
- resolve the allowlisted base model and verify the checkpoint bytes by SHA-256 before use;
- prepare a deterministic synthetic sample **or** upload your own image folder (gated, off by default);
- validate the dataset standalone in-notebook and read its handoff;
- fine-tune the model 100% in-kernel and read its evaluation report;
- compare the model with a majority-class baseline on the frozen validation split;
- reload the exported `safetensors` artifact from a **fresh directory** and prove it reproduces reported metrics;
- classify a new image and export predictions, metrics and provenance as JSON/CSV.

**This notebook does not demonstrate:** ImageNet benchmark reproduction, calibrated probabilities, CPU training, object detection, semantic segmentation, DIMER serving deployment, or production fitness. Metrics on the default synthetic sample are **sanity evidence only**.

Related: [repository README](https://github.com/kurtvalcorza/swin-classification-pipeline#readme) · [tutorial registry](https://github.com/kurtvalcorza/swin-classification-pipeline/blob/main/tutorials/README.md) · [SwinV2 paper](https://arxiv.org/abs/2111.09883) · [`timm/swinv2_tiny_window8_256.ms_in1k`](https://huggingface.co/timm/swinv2_tiny_window8_256.ms_in1k)


## Prerequisites

- **Runtime:** Python 3.11+ with an NVIDIA GPU exposed as `cuda:0` (Colab **T4** or better; Kaggle T4 also works). The finetuner is **fail-closed**: it refuses to run without the expected accelerator instead of silently falling back to CPU. PyTorch/torchvision come from the runtime image and are reported below; the notebook pins the runtime dependencies.
- **Knowledge:** basic Python, the `train/<class>/` image-folder convention, and what accuracy / cross-entropy mean.
- **Data:** the default path generates a deterministic synthetic two-class sample (24 PNGs) and needs no private data. A gated BYOD path accepts a ZIP of your own image folder (schema in Section 3).
- **External access:** GitHub (pipeline release manifest) and the Hugging Face Hub (download the pinned checkpoint). No other service is contacted; uploaded data stays inside this runtime.
- **Credentials:** none. The pipeline repository is **public**, so pinned sources are cloned anonymously. A `GITHUB_TOKEN` secret (Colab Secret, Kaggle secret, or environment variable) is consulted **only if an anonymous clone is refused** — for example a private fork or mirror. When used, the token is passed to Git through an ephemeral HTTP header — it is never printed, embedded in a URL, written to Git config, or exported.
- **Time and memory:** on the default sample the whole notebook is dominated by installs and the 114,918,618-byte (≈115 MB) checkpoint download; training one epoch on 16 images takes seconds on a T4-class GPU.

> **Do not upload confidential or restricted images** to a runtime you are not authorised to use for that data.


## 1. Bootstrap immutable sources and the pinned runtime

Everything downstream is anchored to `PIPELINE_REF`, a specific commit of the pipeline repository. From that commit the notebook reads `pipeline-manifest.json` and the release files, which name the **exact validator and finetuner release commits** the release was composed from.

Both dataset validation and supervised gradient fine-tuning execute **100% in-kernel** without cloning external worker repositories or invoking external CLI subprocesses. Pinned runtime packages (`timm==1.0.28`, `Pillow==12.3.0`, `huggingface_hub==1.29.0`, `safetensors==0.8.0`) are installed directly.

**Look for:** a JSON block naming the source revisions and the runtime (Python, torch, torchvision, timm, CUDA, GPU). If the GPU line is `null`, switch to a GPU runtime and run again.


In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import tempfile
import unicodedata
import uuid
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path, PurePosixPath
from typing import Any

# --- Immutable anchors -----------------------------------------------------------------------
PIPELINE_REPO = "https://github.com/kurtvalcorza/swin-classification-pipeline.git"
PIPELINE_REF = "3e4087ab03d11a60f7ca4d54c4773e31fcfb188a"

WORK = Path("/content/dimer-swin-classification")
PIPELINE_DIR = WORK / "pipeline"
WORK.mkdir(parents=True, exist_ok=True)

PINNED = {"timm": "1.0.28", "Pillow": "12.3.0", "huggingface_hub": "1.29.0", "safetensors": "0.8.0"}


def run(command, cwd=None, env=None):
    """Run a subprocess, echo the command, and raise on non-zero exit."""
    command = [str(part) for part in command]
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True, env=env)


def resolve_github_token():
    """Return a read token for the pipeline repository, or raise a clear error.

    Only called when an anonymous clone was refused (the repository is public by default).

    Resolution order: GITHUB_TOKEN environment variable, Colab Secrets, Kaggle Secrets.
    The value is only ever handed to Git as an HTTP header (see private_git_env)."""
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:  # Colab
            from google.colab import userdata
            token = (userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        try:  # Kaggle (Add-ons > Secrets, attached to the notebook)
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        raise RuntimeError(
            "Anonymous clone of the pipeline repository was refused and no GITHUB_TOKEN secret is available. "
            "The pinned repository is public; if you are using a private fork or mirror, add a GITHUB_TOKEN "
            "secret (Colab: key icon in the left sidebar; Kaggle: Add-ons > Secrets) with read access to "
            "kurtvalcorza/swin-classification-pipeline, then run this cell again."
        )
    return token


def private_git_env(token):
    """Git environment that authenticates via an ephemeral extraHeader (never a URL, never config on disk)."""
    credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {credential}",
    })
    return env


def checkout_pinned(url, ref, dst, env=None):
    """Clone (blobless) if needed, then detach at the exact pinned commit."""
    if not dst.exists():
        run(["git", "clone", "--filter=blob:none", url, dst], env=env)
    run(["git", "checkout", "--detach", ref], cwd=dst, env=env)
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dst, check=True, capture_output=True, text=True).stdout.strip()
    if head != ref:
        raise RuntimeError(f"{dst.name}: checked out {head}, expected {ref}")


# --- Public pipeline repository at the immutable anchor --------------------------------------
if not PIPELINE_DIR.exists():
    try:
        checkout_pinned(PIPELINE_REPO, PIPELINE_REF, PIPELINE_DIR)
        pipeline_source_mode = "cloned anonymously from public GitHub; no credential used"
    except subprocess.CalledProcessError as clone_error:
        try:
            PRIVATE_TOKEN = resolve_github_token()
        except RuntimeError as no_token:
            raise RuntimeError(f"{no_token} (anonymous clone failed with: {clone_error})") from clone_error
        PRIVATE_GIT_ENV = private_git_env(PRIVATE_TOKEN)
        checkout_pinned(PIPELINE_REPO, PIPELINE_REF, PIPELINE_DIR, env=PRIVATE_GIT_ENV)
        del PRIVATE_TOKEN, PRIVATE_GIT_ENV
        pipeline_source_mode = "cloned from GitHub with an ephemeral read token (anonymous clone refused)"
else:
    checkout_pinned(PIPELINE_REPO, PIPELINE_REF, PIPELINE_DIR)
    pipeline_source_mode = "pre-staged checkout found; no credential used"
print("pipeline source:", pipeline_source_mode)

pipeline_manifest = json.loads((PIPELINE_DIR / "pipeline-manifest.json").read_text())
validator_release = json.loads((PIPELINE_DIR / "release/worker-release-validator.json").read_text())
finetuner_release = json.loads((PIPELINE_DIR / "release/worker-release-finetuner.json").read_text())
VALIDATOR_REF = validator_release["sourceRevision"]
FINETUNER_REF = finetuner_release["sourceRevision"]

# --- Pinned runtime packages -----------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.28", "huggingface_hub==1.29.0", "safetensors==0.8.0", "pillow==12.3.0"])
import importlib
importlib.invalidate_caches()

import PIL
import timm
import torch
import torchvision
from PIL import Image, ImageDraw, ImageOps, UnidentifiedImageError

# Preprocessing constants
NORMALIZATION_MEAN = (0.485, 0.456, 0.406)
NORMALIZATION_STD = (0.229, 0.224, 0.225)
EXIF_ORIENTATION = "transpose-to-visual-orientation"
EVAL_INTERPOLATION = "bicubic"

def load_visual_image(path: Path):
    """Decode with the declared EXIF semantic: visual orientation."""
    with Image.open(path) as image:
        image.load()
        return ImageOps.exif_transpose(image).convert("RGB")

# Restart-boundary check
if timm.__version__ != PINNED["timm"]:
    raise RuntimeError(f"timm {timm.__version__} is active but {PINNED['timm']} was installed. Restart the runtime (Runtime > Restart session) and run from the top.")
if PIL.__version__ != PINNED["Pillow"]:
    print(f"WARNING: this kernel had Pillow {PIL.__version__} imported before the pinned {PINNED['Pillow']} was installed. "
          "Restart the runtime for exact in-kernel parity.")

runtime = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "pillowInKernel": PIL.__version__,
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps({
    "sources": {
        "pipeline": PIPELINE_REF,
        "pipelineSourceMode": pipeline_source_mode,
        "validator": f"{VALIDATOR_REF} (standalone in-notebook)",
        "finetuner": f"{FINETUNER_REF} (100% in-kernel)",
    },
    "runtime": runtime,
}, indent=2))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required by the fail-closed finetuner. In Colab choose Runtime > Change runtime type > GPU, then Run all.")


## 2. Resolve the pinned base model

The pipeline maintains an allowlisted **base-model catalog** mapping each catalog key to a Hugging Face repository, an immutable commit revision, the exact file, its SHA-256, license, and qualification status. Off-catalog models are refused. The notebook stages the file and verifies the SHA-256 digest before building the model.


In [ ]:
MODEL_KEY = "swinv2-tiny-window8-256-ms-in1k"  # @param ["swinv2-tiny-window8-256-ms-in1k", "swinv2-small-window8-256-ms-in1k"]

BASE_MODEL_CATALOG = {
    "schemaVersion": "1.0",
    "catalogVersion": "0.2",
    "entries": {
        "swinv2-tiny-window8-256-ms-in1k": {
            "modelDescriptorId": "org.timm.swinv2-tiny-window8-256-ms-in1k",
            "modelDescriptorVersion": "650d02aabf05e8adbd060a739ab39e39f53da639",
            "timmModelName": "swinv2_tiny_window8_256.ms_in1k",
            "source": {
                "repoId": "timm/swinv2_tiny_window8_256.ms_in1k",
                "revision": "650d02aabf05e8adbd060a739ab39e39f53da639",
                "files": [
                    {"path": "model.safetensors", "digest": "sha256:c47f52b4556ff4436aa9502f5efbc93aac77fdab42d9d70dd845931757ff5d65"}
                ]
            },
            "license": {"spdx": "MIT", "datasetTerms": "ImageNet-1k dataset terms apply to the pretrained weights."},
            "input": {"height": 256, "width": 256},
            "qualification": {"status": "QUALIFIED"}
        },
        "swinv2-small-window8-256-ms-in1k": {
            "modelDescriptorId": "org.timm.swinv2-small-window8-256-ms-in1k",
            "modelDescriptorVersion": "0c9500fcde4c689e97ff51954debae59c158af0d",
            "timmModelName": "swinv2_small_window8_256.ms_in1k",
            "source": {
                "repoId": "timm/swinv2_small_window8_256.ms_in1k",
                "revision": "0c9500fcde4c689e97ff51954debae59c158af0d",
                "files": [
                    {"path": "model.safetensors", "digest": "sha256:7e793c2f3576d20b5f746ac5dc6b7271745d612f9ce06ccabe382c36f39f1caa"}
                ]
            },
            "license": {"spdx": "MIT", "datasetTerms": "ImageNet-1k dataset terms apply to the pretrained weights."},
            "input": {"height": 256, "width": 256},
            "qualification": {"status": "QUALIFIED"}
        }
    }
}

if MODEL_KEY not in BASE_MODEL_CATALOG["entries"]:
    raise RuntimeError(f"{MODEL_KEY!r} is not an allowlisted catalog entry: {sorted(BASE_MODEL_CATALOG['entries'])}")
catalog_entry = BASE_MODEL_CATALOG["entries"][MODEL_KEY]
weight_file = catalog_entry["source"]["files"][0]
model_identity = {
    "modelKey": MODEL_KEY,
    "modelDescriptorId": catalog_entry["modelDescriptorId"],
    "modelDescriptorVersion": catalog_entry["modelDescriptorVersion"],
    "timmModelName": catalog_entry["timmModelName"],
    "repoId": catalog_entry["source"]["repoId"],
    "revision": catalog_entry["source"]["revision"],
    "file": weight_file["path"],
    "expectedSha256": weight_file["digest"],
    "license": catalog_entry["license"],
    "qualificationStatus": catalog_entry["qualification"]["status"],
}
print(json.dumps(model_identity, indent=2))


## 3. Prepare the dataset: default synthetic sample or bring your own

### Expected input schema (image-folder representation)

```
<dataset root>/
  train/<class name>/*.png|jpg|jpeg|bmp|tif|tiff|webp
  val/<class name>/...          (or valid/ — exactly one of the two)
  test/<class name>/...         (optional; never reinterpreted as validation)
```

Class names are directory names; two names that collide after Unicode normalisation and case folding (`Cat` and `cat`) are refused as a collision, not treated as two classes. Every class should appear in both `train/` and `val/` (the validator warns otherwise, and the evaluation would be meaningless). The dataset root may contain **only** the split directories, and split directories only class directories of images: stray files, nested folders, symlinks, unrecognised extensions and undecodable images are **fatal** validator findings — nothing is silently skipped or substituted. Any decodable image is converted to 3-channel RGB and resized to 256×256 **without preserving aspect ratio**; there is no channel-count check and no image-count ceiling — GPU memory and the batch size are the practical limits.

### Default path (runs without any upload)

`USE_BYOD_DATASET = False` generates 24 deterministic 256×256 PNGs from a fixed seed: class `cool` (blue background, white square) and class `warm` (red background, cream circle), 8 training and 4 validation images per class. This is **smoke data**: it proves the plumbing and cannot say anything about real-world accuracy.

### BYOD path (gated)

Set `USE_BYOD_DATASET = True` and run the cell; in Colab an upload dialog asks for **one ZIP** laid out as above (a single top-level folder inside the ZIP is fine). The archive is extracted with path-safety checks — absolute paths, `..`, backslash-ambiguous names, symlinks, anything escaping the extraction root, or an expanded size above `MAX_EXPANDED_MIB` are rejected. Your images stay in this runtime and are only read by the local validator and finetuner. Outside Colab, set `BYOD_ZIP_PATH` instead of using the dialog.

**Look for:** a small table of images per split and class. Every class should have at least one image in both `train` and `val`.


In [ ]:
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ""  # @param {type:"string"}
MAX_EXPANDED_MIB = 2048  # @param {type:"integer"}
SAMPLE_SEED = 20260910  # @param {type:"integer"}

DATASET_DIR = WORK / "dataset"
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def safe_extract_zip(zip_path: Path, destination: Path, max_expanded_bytes: int) -> int:
    """Extract an image-folder ZIP with the archive-safety rules of DIMER Notebook Spec section 19."""
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    expanded = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            name = info.filename
            posix = PurePosixPath(name)
            if "\\" in name:
                raise ValueError(f"backslash-ambiguous archive member rejected: {name!r}")
            if posix.is_absolute() or name.startswith("/") or ".." in posix.parts:
                raise ValueError(f"absolute or traversing archive member rejected: {name!r}")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f"symlink archive member rejected: {name!r}")
            expanded += info.file_size
            if expanded > max_expanded_bytes:
                raise ValueError(f"archive expands beyond {max_expanded_bytes} bytes; raise MAX_EXPANDED_MIB only if you trust the file")
            target = (root / posix).resolve()
            if root != target and root not in target.parents:
                raise ValueError(f"archive member escapes extraction root: {name!r}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    return expanded


def locate_dataset_root(extracted: Path) -> Path:
    """Accept either `train/` at the top level or exactly one wrapper directory around it."""
    if (extracted / "train").is_dir():
        return extracted
    children = [p for p in extracted.iterdir() if p.is_dir() and not p.name.startswith(("__MACOSX", "."))]
    if len(children) == 1 and (children[0] / "train").is_dir():
        return children[0]
    raise ValueError("ZIP must contain train/ (and val/ or valid/) at the top level or inside one wrapper folder")


if USE_BYOD_DATASET:
    if BYOD_ZIP_PATH:
        zip_path = Path(BYOD_ZIP_PATH)
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise RuntimeError("Not running in Colab: set BYOD_ZIP_PATH to a ZIP already present in this runtime.") from error
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Upload exactly one ZIP file.")
        zip_path = WORK / next(iter(uploaded))
        zip_path.write_bytes(next(iter(uploaded.values())))
    extracted = WORK / "byod-extracted"
    if extracted.exists():
        shutil.rmtree(extracted)
    expanded = safe_extract_zip(zip_path, extracted, MAX_EXPANDED_MIB * 1024 * 1024)
    shutil.copytree(locate_dataset_root(extracted), DATASET_DIR)
    dataset_origin = {"type": "user-supplied image folder (BYOD)", "zip": zip_path.name, "expandedBytes": expanded}
else:
    rng = random.Random(SAMPLE_SEED)
    for split, per_class in {"train": 8, "val": 4}.items():
        for class_name in ("cool", "warm"):
            out = DATASET_DIR / split / class_name
            out.mkdir(parents=True, exist_ok=True)
            for index in range(per_class):
                background = (35, 70, 190) if class_name == "cool" else (190, 65, 35)
                image = Image.new("RGB", (256, 256), background)
                draw = ImageDraw.Draw(image)
                jitter = rng.randint(-15, 15)
                if class_name == "cool":
                    draw.rectangle((60 + jitter, 60, 196 + jitter, 196), outline=(230, 240, 255), width=10)
                else:
                    draw.ellipse((60 + jitter, 60, 196 + jitter, 196), outline=(255, 240, 220), width=10)
                image.save(out / f"{class_name}-{index:02d}.png")
    dataset_origin = {"type": "deterministic synthetic tutorial sample", "seed": SAMPLE_SEED, "generator": "this notebook, Section 3"}

# Compact inventory so you can sanity-check the layout before validation.
inventory = {}
for path in sorted(DATASET_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        split, class_name = path.relative_to(DATASET_DIR).parts[:2]
        inventory.setdefault(class_name, {}).setdefault(split, 0)
        inventory[class_name][split] += 1
splits = sorted({s for counts in inventory.values() for s in counts})
print(f"{'class':<24}" + "".join(f"{s:>8}" for s in splits))
for class_name, counts in inventory.items():
    print(f"{class_name:<24}" + "".join(f"{counts.get(s, 0):>8}" for s in splits))
print(json.dumps(dataset_origin, indent=2))


## 4. Standalone dataset validation and handoff

Dataset validation is executed directly in this notebook as a standalone stage. It decodes every image (decode failure is fatal), maps `train/` → `train`, `val/`|`valid/` → `validation`, `test/` → `test` without ever reinterpreting a split, assigns a stable sample ID (the relative path) to every file, computes SHA-256 digests for every file, and writes a complete DIMER **handoff** of documents: `logical-dataset-manifest.json`, `data-plan.json`, `semantic-dataset-schema.json`, and `validated-dataset-manifest.json`.

The downstream finetuner consumes that handoff *as is* — verifying every digest and enforcing that no samples are re-split, dropped, or mutated. Because the validation runs in-notebook using standard libraries and Pillow, end users can run, export, or download local copies without requiring an external validator repository or CLI.

In DIMER the `--*-digest` bindings bind the run to the platform's job spec, admission record and security grant. In this tutorial they are deterministic stand-ins derived from fixed content, so the run is reproducible and adheres strictly to the DIMER worker contract.

**Look for:** `"state": "SUCCEEDED"`, the frozen `labelMap` (class index → class name), split counts, and any `WARNING` findings (for example a class missing from a required split).


In [ ]:
TASK = "core.task.vision.image-classification"
REP = "core.dataset.vision.image-folder"
ALG = "org.valcorza.swin-classification-validator.v1"
EXT = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".bmp": "image/bmp", ".tif": "image/tiff", ".tiff": "image/tiff", ".webp": "image/webp"}
SPLITS = {"train": "train", "val": "validation", "valid": "validation", "test": "test"}


class ValidationFailure(RuntimeError):
    def __init__(self, finding: dict):
        super().__init__(finding["message"])
        self.finding = finding


def cbytes(v):
    return json.dumps(v, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()


def dbytes(v: bytes) -> str:
    return "sha256:" + hashlib.sha256(v).hexdigest()


def djson(v) -> str:
    return dbytes(cbytes(v))


def fdigest(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return "sha256:" + h.hexdigest()


def atomic_write_json(path: Path, v) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    data = cbytes(v) + b"\n"
    fd, tmp = tempfile.mkstemp(prefix="." + path.name + ".", dir=path.parent)
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.unlink(tmp)
    return dbytes(data[:-1])


def finding(code, severity, layer, scope, message, observed, expected, evidence=None):
    return {
        "code": code, "severity": severity, "layer": layer, "scope": scope,
        "location": None, "message": message, "observed": observed,
        "expected": expected, "evidence": evidence or {},
    }


def fail(code, layer, scope, message, observed, expected, **evidence):
    raise ValidationFailure(finding(code, "FATAL", layer, scope, message, observed, expected, evidence))


def safe_rel(root: Path, p: Path) -> str:
    if p.is_symlink():
        fail("VISION_SYMLINK_REJECTED", "L0", "dataset.path", "Symbolic links are forbidden.", str(p), "regular path")
    try:
        return p.resolve(strict=True).relative_to(root).as_posix()
    except ValueError:
        fail("VISION_PATH_ESCAPE", "L0", "dataset.path", "Path resolves outside dataset root.", str(p), str(root))


def root_checked(root: Path) -> Path:
    if not root.exists():
        fail("VISION_DATASET_ROOT_MISSING", "L0", "dataset.root", "Dataset root does not exist.", str(root), "existing directory")
    if root.is_symlink():
        fail("VISION_SYMLINK_REJECTED", "L0", "dataset.root", "Dataset root must not be a symlink.", str(root), "non-symlink directory")
    if not root.is_dir():
        fail("VISION_DATASET_ROOT_NOT_DIRECTORY", "L0", "dataset.root", "Dataset root is not a directory.", str(root), "directory")
    return root.resolve(strict=True)


def source_digest(root: Path) -> str:
    rows = []
    for cur, dirs, files in os.walk(root, topdown=True, followlinks=False):
        dirs.sort()
        files.sort()
        cp = Path(cur)
        for n in dirs:
            safe_rel(root, cp / n)
        for n in files:
            p = cp / n
            rows.append({"path": safe_rel(root, p), "digest": fdigest(p)})
    return djson({"algorithmId": ALG + ".source-snapshot", "files": rows})


def decode_image_file(p: Path, media: str):
    try:
        with Image.open(p) as im:
            fmt = im.format
            im.verify()
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im)
            im.load()
    except (UnidentifiedImageError, OSError, ValueError) as e:
        fail("VISION_IMAGE_DECODE_FAILED", "L1", "dataset.image", "Image decode failed; synthetic replacement pixels are forbidden.", str(p), "decodable image", error=type(e).__name__)
    actual = {"JPEG": "image/jpeg", "PNG": "image/png", "BMP": "image/bmp", "TIFF": "image/tiff", "WEBP": "image/webp"}.get(fmt or "")
    if actual != media:
        fail("VISION_IMAGE_EXTENSION_FORMAT_MISMATCH", "L1", "dataset.image", "Decoded image format contradicts extension.", actual, media, path=str(p))


def inspect_dataset(root: Path) -> dict:
    root = root_checked(root)
    src = source_digest(root)
    entries = sorted(root.iterdir(), key=lambda p: p.name)
    unknown = [p.name for p in entries if p.name not in SPLITS]
    if unknown:
        fail("VISION_UNKNOWN_ROOT_ENTRY", "L1", "dataset.structure", "Dataset root contains unsupported entries.", unknown, sorted(SPLITS))
    present = {p.name for p in entries if p.is_dir()}
    if "val" in present and "valid" in present:
        fail("VISION_AMBIGUOUS_VALIDATION_SPLIT", "L1", "dataset.splits", "Both val/ and valid/ exist; no precedence is defined.", ["val", "valid"], "exactly one validation directory")
    if "train" not in present:
        fail("VISION_MISSING_TRAIN_SPLIT", "L2", "dataset.splits", "train/ is required.", sorted(present), "train present")
    if not ({"val", "valid"} & present):
        fail("VISION_MISSING_VALIDATION_SPLIT", "L2", "dataset.splits", "A validation split is required; test/ is never reinterpreted as validation.", sorted(present), "val/ or valid/ present")

    samples, assets, assignments = [], [], []
    classes = set()
    class_presence = {}
    warnings = []

    for sd in ("train", "val", "valid", "test"):
        sp = root / sd
        if not sp.exists():
            continue
        safe_rel(root, sp)
        logical = SPLITS[sd]
        cdirs = sorted(sp.iterdir(), key=lambda p: p.name)
        if not cdirs:
            fail("VISION_EMPTY_SPLIT", "L1", "dataset.splits", "Split directory is empty.", sd, "one or more class directories")
        seen = {}
        for cd in cdirs:
            safe_rel(root, cd)
            if not cd.is_dir():
                fail("VISION_UNASSIGNED_SPLIT_FILE", "L1", "dataset.structure", "Files directly inside split directories are not assignable to a class.", cd.name, "class directory")
            label = cd.name
            key = unicodedata.normalize("NFC", label).casefold()
            if key in seen and seen[key] != label:
                fail("VISION_CLASS_NAME_COLLISION", "L2", "dataset.classes", "Class names collide under normalization/case folding.", [seen[key], label], "distinct normalized labels")
            seen[key] = label
            classes.add(label)
            class_presence.setdefault(label, set()).add(logical)
            ims = sorted(cd.iterdir(), key=lambda p: p.name)
            if not ims:
                fail("VISION_EMPTY_CLASS_DIRECTORY", "L1", "dataset.classes", "Class directory is empty.", cd.as_posix(), "one or more image files")
            for p in ims:
                rel = safe_rel(root, p)
                if p.is_dir():
                    fail("VISION_NESTED_CLASS_DIRECTORY", "L1", "dataset.structure", "Nested directories below a class directory are forbidden.", rel, "image file")
                media = EXT.get(p.suffix.lower())
                if not media:
                    fail("VISION_UNRECOGNIZED_IMAGE_EXTENSION", "L1", "dataset.image", "Unrecognized image extension; files are never silently skipped.", p.suffix.lower(), sorted(EXT), path=rel)
                decode_image_file(p, media)
                dg = fdigest(p)
                samples.append({"sampleId": rel, "assetIds": [rel], "sourceLocator": rel})
                assets.append({"assetId": rel, "digest": dg, "mediaType": media})
                assignments.append({"sampleId": rel, "split": logical, "reason": "directory-mapping"})

    if not samples:
        fail("VISION_NO_SAMPLES", "L1", "dataset.samples", "No image samples were discovered.", 0, ">= 1")

    seen = {}
    for sid in [s["sampleId"] for s in samples]:
        k = unicodedata.normalize("NFC", sid).casefold()
        if k in seen and seen[k] != sid:
            fail("VISION_SAMPLE_ID_COLLISION", "L1", "dataset.samples", "Logical sample IDs collide after canonicalization.", [seen[k], sid], "unique canonical sample IDs")
        seen[k] = sid

    amap = {a["sampleId"]: a["split"] for a in assignments}
    dig = {a["assetId"]: a["digest"] for a in assets}
    logical_rows = [
        {"sampleId": s["sampleId"], "split": amap[s["sampleId"]], "className": Path(s["sampleId"]).parts[1], "contentDigest": dig[s["sampleId"]]}
        for s in sorted(samples, key=lambda x: x["sampleId"])
    ]
    logical = djson({"algorithmId": ALG + ".logical-dataset", "representationProfile": REP, "samples": logical_rows})
    cls = sorted(classes)
    semantic = {
        "schemaVersion": "1.0",
        "taskProfile": TASK,
        "fields": [
            {"id": "image", "sourceField": None, "logicalType": "core.type.image", "semanticRole": "core.role.image", "nullable": False},
            {"id": "target.class", "sourceField": None, "logicalType": "core.type.class-label", "semanticRole": "core.role.target.class", "nullable": False},
        ],
        "labelMap": {str(i): n for i, n in enumerate(cls)},
    }
    plan = {
        "schemaVersion": "1.0",
        "logicalDatasetDigest": logical,
        "assignments": sorted(assignments, key=lambda x: x["sampleId"]),
        "seedPolicy": None,
    }
    manifest = {
        "schemaVersion": "1.0",
        "sourceArtifactDigest": src,
        "representationProfile": REP,
        "logicalDatasetDigest": logical,
        "samples": sorted(samples, key=lambda x: x["sampleId"]),
        "assets": sorted(assets, key=lambda x: x["assetId"]),
    }

    for c in cls:
        missing = sorted({"train", "validation"} - class_presence.get(c, set()))
        if missing:
            warnings.append(finding(
                "VISION_CLASS_ABSENT_FROM_REQUIRED_SPLIT", "WARNING", "L3", "dataset.classes",
                "A class is absent from one or more required splits.",
                {"className": c, "missingSplits": missing}, "class represented in train and validation",
                {"className": c},
            ))

    by_digest = {}
    for a in assets:
        by_digest.setdefault(a["digest"], []).append(a["assetId"])
    leaked = []
    for dg, ids in sorted(by_digest.items()):
        splits = sorted({amap[i] for i in ids})
        if len(splits) > 1:
            leaked.append({"contentDigest": dg, "sampleIds": sorted(ids), "splits": splits})
    if leaked:
        warnings.append(finding(
            "VISION_DUPLICATE_CONTENT_ACROSS_SPLITS", "WARNING", "L3", "dataset.splits",
            "Byte-identical images appear in more than one split; held-out metrics on those samples are not independent evidence.",
            {"duplicateGroups": len(leaked), "groups": leaked}, "each image content present in at most one split",
        ))

    return {
        "sourceArtifactDigest": src,
        "logicalDatasetDigest": logical,
        "logicalManifest": manifest,
        "semanticSchema": semantic,
        "dataPlan": plan,
        "warnings": warnings,
        "classNames": cls,
    }


def validate_dataset(root: Path, out: Path, worker_release_digest: str, job_id: str = "tutorial-classification", attempt_id: str = "attempt-1", effective_job_spec_digest: str = "", admission_record_digest: str = "", security_grant_digest: str = ""):
    """Standalone in-notebook dataset validator producing verified DIMER handoff artifacts."""
    out.mkdir(parents=True, exist_ok=True)
    cfg = {
        "algorithmId": ALG, "taskProfile": TASK, "representationProfile": REP,
        "acceptedImageExtensions": sorted(EXT), "exifOrientation": "transpose-to-visual-orientation",
        "splitMapping": SPLITS, "requireTrain": True, "requireValidation": True, "testAsValidation": False,
    }
    cfgd = djson(cfg)
    ep = {
        "schemaVersion": "1.0", "jobId": job_id, "attemptId": attempt_id, "role": "validator",
        "workerReleaseDigest": worker_release_digest, "effectiveJobSpecDigest": effective_job_spec_digest,
        "resourceBindingDigests": [], "admissionRecordDigest": admission_record_digest,
        "securityGrantDigest": security_grant_digest,
    }
    epd = atomic_write_json(out / "execution-plan.json", ep)

    try:
        x = inspect_dataset(root)
    except ValidationFailure as e:
        atomic_write_json(out / "validation-findings.json", {"schemaVersion": "1.0", "algorithmId": ALG, "findings": [e.finding]})
        rm = {
            "schemaVersion": "1.0", "jobId": job_id, "attemptId": attempt_id, "executionPlanDigest": epd,
            "workerReleaseDigest": worker_release_digest, "outcome": "FAILED",
            "observed": {"findingCodes": [e.finding["code"]]}, "artifactManifestDigest": None,
            "evaluationReportDigest": None, "reproducibility": "IDENTIFIED",
        }
        rmd = atomic_write_json(out / "run-manifest.json", rm)
        atomic_write_json(out / "result.json", {
            "schemaVersion": "1.0", "jobId": job_id, "attemptId": attempt_id, "state": "FAILED",
            "failure": {"code": e.finding["code"], "category": "INPUT", "retryable": False, "origin": ALG, "details": {"finding": e.finding}},
            "runManifestDigest": rmd, "artifactManifestDigest": None,
        })
        raise

    lmd = atomic_write_json(out / "logical-dataset-manifest.json", x["logicalManifest"])
    ssd = atomic_write_json(out / "semantic-dataset-schema.json", x["semanticSchema"])
    dpd = atomic_write_json(out / "data-plan.json", x["dataPlan"])

    ed, prev = [], []
    for layer, finds in [("L0", []), ("L1", []), ("L2", []), ("L3", x["warnings"]), ("L4", [])]:
        ev = {
            "schemaVersion": "1.0", "logicalDatasetDigest": x["logicalDatasetDigest"],
            "layer": layer, "algorithmId": ALG + "." + layer.lower(),
            "outcome": "FAIL" if any(f["severity"] in {"ERROR", "FATAL"} for f in finds) else "PASS",
            "findings": finds, "dependencies": list(prev), "cacheability": "REUSABLE",
        }
        dg = atomic_write_json(out / "evidence" / (layer.lower() + ".json"), ev)
        ed.append(dg)
        prev = [dg]

    vm = {
        "schemaVersion": "1.0", "logicalDatasetDigest": x["logicalDatasetDigest"],
        "datasetProfileDigest": None, "semanticSchemaDigest": ssd, "dataPlanDigest": dpd,
        "validationEvidenceDigests": ed, "validatorWorkerReleaseDigest": worker_release_digest,
        "effectiveValidationConfigDigest": cfgd, "policySetDigest": None,
    }
    vmd = atomic_write_json(out / "validated-dataset-manifest.json", vm)

    vi = {
        "schemaVersion": "1.0", "algorithmId": ALG + ".validated-dataset-identity",
        "validatedDatasetManifestDigest": vmd, "logicalDatasetDigest": x["logicalDatasetDigest"],
        "effectiveValidationConfigDigest": cfgd,
        "digest": djson({"manifest": vmd, "logical": x["logicalDatasetDigest"], "config": cfgd, "worker": worker_release_digest, "semantic": ssd, "dataPlan": dpd, "evidence": ed}),
    }
    vid = atomic_write_json(out / "validated-dataset-identity.json", vi)

    rm = {
        "schemaVersion": "1.0", "jobId": job_id, "attemptId": attempt_id, "executionPlanDigest": epd,
        "workerReleaseDigest": worker_release_digest, "outcome": "SUCCEEDED",
        "observed": {
            "sourceArtifactDigest": x["sourceArtifactDigest"], "logicalDatasetManifestDigest": lmd,
            "validatedDatasetManifestDigest": vmd, "validatedDatasetIdentityDigest": vid,
            "sampleCount": len(x["logicalManifest"]["samples"]), "classNames": x["classNames"],
        },
        "artifactManifestDigest": None, "evaluationReportDigest": None, "reproducibility": "REEXECUTABLE",
    }
    rmd = atomic_write_json(out / "run-manifest.json", rm)

    atomic_write_json(out / "result.json", {
        "schemaVersion": "1.0", "jobId": job_id, "attemptId": attempt_id,
        "state": "SUCCEEDED", "failure": None, "runManifestDigest": rmd, "artifactManifestDigest": None,
    })
    return {
        "validatedDatasetManifestDigest": vmd, "validatedDatasetIdentityDigest": vid,
        "logicalDatasetDigest": x["logicalDatasetDigest"], "dataPlanDigest": dpd,
        "semanticDatasetSchemaDigest": ssd, "runManifestDigest": rmd,
    }


def digest_json(document) -> str:
    """Canonical JSON SHA-256 in the `sha256:<hex>` form the workers use."""
    encoded = json.dumps(document, sort_keys=True, separators=(",", ":")).encode()
    return "sha256:" + hashlib.sha256(encoded).hexdigest()


binding = {
    "job": digest_json({"kind": "tutorial-job", "task": pipeline_manifest["taskProfile"]}),
    "admission": digest_json({"kind": "tutorial-admission"}),
    "security": digest_json({"kind": "tutorial-security", "networkDuringRunning": "DENY"}),
}
HANDOFF_DIR = WORK / "validated"
if HANDOFF_DIR.exists():
    shutil.rmtree(HANDOFF_DIR)

validate_dataset(
    DATASET_DIR,
    HANDOFF_DIR,
    worker_release_digest=pipeline_manifest["validatorWorkerReleaseDigest"],
    job_id="tutorial-classification",
    attempt_id="attempt-1",
    effective_job_spec_digest=binding["job"],
    admission_record_digest=binding["admission"],
    security_grant_digest=binding["security"],
)

validation_result = json.loads((HANDOFF_DIR / "result.json").read_text())
if validation_result.get("state") != "SUCCEEDED":
    findings_path = HANDOFF_DIR / "validation-findings.json"
    detail = findings_path.read_text() if findings_path.exists() else json.dumps(validation_result, indent=2)
    raise RuntimeError("Dataset validation failed. Fix the dataset layout and rerun Section 3 and 4.\n" + detail)

semantic_schema = json.loads((HANDOFF_DIR / "semantic-dataset-schema.json").read_text())
data_plan = json.loads((HANDOFF_DIR / "data-plan.json").read_text())
logical_manifest = json.loads((HANDOFF_DIR / "logical-dataset-manifest.json").read_text())
label_map = semantic_schema["labelMap"]
class_names = [label_map[key] for key in sorted(label_map, key=int)]  # index -> class name, frozen by the validator
split_counts = {}
for assignment in data_plan["assignments"]:
    split_counts[assignment["split"]] = split_counts.get(assignment["split"], 0) + 1
findings_path = HANDOFF_DIR / "validation-findings.json"
warnings = json.loads(findings_path.read_text()).get("findings", []) if findings_path.exists() else []

print(json.dumps({
    "state": validation_result["state"],
    "logicalDatasetDigest": logical_manifest["logicalDatasetDigest"],
    "labelMap": label_map,
    "splitCounts": split_counts,
    "warnings": [{"code": w.get("code"), "detail": w.get("observed")} for w in warnings],
}, indent=2))
if split_counts.get("validation", 0) == 0:
    raise RuntimeError("No validation samples were assigned; the evaluation below would be empty.")


## 5. Stage and SHA-256 verify the pinned checkpoint

The checkpoint is downloaded from the Hugging Face repository **at the immutable revision** recorded in the catalog, copied into the layout the finetuner expects (`<weights root>/<model key>/<file>`), and hashed. A digest mismatch deletes the file and stops the notebook — the worker would refuse it anyway, but failing here gives a clearer message.

The file is `model.safetensors`: a tensor-only format with no code execution on load, so no `trust_remote_code` or pickle trust decision is needed anywhere in this pipeline.

Digest equality proves the bytes are the ones the catalog allowlisted; it does **not** by itself prove who published them — that trust rests on the catalog review that produced the pin.

**Look for:** `expected` and `observed` digests that are identical.


In [ ]:
from huggingface_hub import hf_hub_download

WEIGHTS_DIR = WORK / "weights"
staged_file = WEIGHTS_DIR / MODEL_KEY / weight_file["path"]
staged_file.parent.mkdir(parents=True, exist_ok=True)

downloaded = Path(hf_hub_download(
    repo_id=catalog_entry["source"]["repoId"],
    filename=weight_file["path"],
    revision=catalog_entry["source"]["revision"],
))
shutil.copyfile(downloaded, staged_file)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return "sha256:" + digest.hexdigest()


observed_digest = sha256_file(staged_file)
print(json.dumps({"file": str(staged_file), "bytes": staged_file.stat().st_size,
                  "expected": weight_file["digest"], "observed": observed_digest}, indent=2))
if observed_digest != weight_file["digest"]:
    staged_file.unlink(missing_ok=True)
    raise RuntimeError("Checkpoint digest mismatch: the downloaded bytes are not the allowlisted checkpoint. Do not proceed.")


## 6. In-Kernel Supervised Fine-Tuning

Supervised fine-tuning runs **100% inside this notebook kernel**:

1. Requires `--expected-accelerator cuda:0` to match the observed device — **no silent CPU fallback**;
2. Loads the validated dataset splits (`train/` and `validation/`) according to the frozen `data-plan.json`;
3. Replaces the SwinV2 classifier head with one output per class (`num_classes = len(class_names)`), and fine-tunes **all parameters** with AdamW and cross-entropy;
4. Publishes a content-addressed artifact generation (`model.safetensors` + `model-config.json` + `model_manifest.json` — the `dimer-inference-service-timm` serving manifest — + `artifact-manifest.json` with per-member digests);
5. Frees the trained model, reloads the persisted artifact, and evaluates the frozen `validation` split with it — reported metrics come from the persisted bytes, not the in-memory model.

**Look for:** `"state": "SUCCEEDED"`, the per-epoch `history`, and the two evaluation metrics.

Seeds control data order and initialisation of the new head. The run is *re-executable*, not bitwise reproducible: cuDNN kernel selection and floating-point reduction order on the GPU still vary, so expect identical accuracy on this sample but small drift in loss and a different artifact digest between runs.


In [ ]:
EPOCHS = 1  # @param {type:"integer"}
BATCH_SIZE = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
SEED = 20260910  # @param {type:"integer"}

TRAINING_DIR = WORK / "training-output"
if TRAINING_DIR.exists():
    shutil.rmtree(TRAINING_DIR)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ACCELERATOR = "cuda:0"  # @param {type:"string"}
if EXPECTED_ACCELERATOR != "cuda:0":
    raise RuntimeError(f"Expected accelerator must be 'cuda:0', got {EXPECTED_ACCELERATOR!r}")
if not torch.cuda.is_available():
    raise RuntimeError("ACCELERATOR_UNAVAILABLE: No CUDA accelerator observed; silent CPU fallback is forbidden.")

DEVICE = torch.device(EXPECTED_ACCELERATOR)

def build_model_manifest(model_config: dict[str, Any], checkpoint: str = "model.safetensors") -> dict[str, Any]:
    """Derive model_manifest.json (dimer-inference-service-timm contract) from model-config.json."""
    validation = model_config["transforms"]["validation"]
    resize = next(t for t in validation if t["id"] == "org.torchvision.resize")
    normalize = next(t for t in validation if t["id"] == "org.torchvision.normalize")
    height, width = resize["size"]
    class_names = list(model_config["classNames"])
    num_classes = int(model_config["numClasses"])
    return {
        "schema_version": 1,
        "framework": "timm",
        "model": model_config["timmModelName"],
        "num_classes": num_classes,
        "class_names": class_names,
        "checkpoint": checkpoint,
        "use_ema": False,
        "preprocessing": {
            "input_size": [3, int(height), int(width)],
            "mean": [float(v) for v in normalize["mean"]],
            "std": [float(v) for v in normalize["std"]],
            "interpolation": EVAL_INTERPOLATION,
            "crop_pct": 1.0,
            "crop_mode": "squash",
        },
    }


@dataclass(frozen=True)
class ArtifactMemberSource:
    role: str
    path: Path
    media_type: str
    required: bool = True


def publish_artifact_bundle(artifact_root: Path, members: tuple[ArtifactMemberSource, ...]) -> tuple[dict[str, Any], Path]:
    """Atomically publish generation directory and CURRENT pointer."""
    artifact_root = Path(artifact_root)
    generations = artifact_root / "generations"
    generations.mkdir(parents=True, exist_ok=True)
    stage = artifact_root / f".staging-{uuid.uuid4().hex}"
    stage.mkdir()
    try:
        records = []
        for source in members:
            target = stage / source.path.name
            shutil.copyfile(source.path, target)
            records.append({
                "role": source.role,
                "digest": sha256_file(target),
                "mediaType": source.media_type,
                "required": source.required,
                "relationships": [{"type": "org.valcorza.bundle.member-path", "path": source.path.name}],
            })
        records.sort(key=lambda item: item["role"])
        manifest = {"schemaVersion": "1.0", "members": records}
        bundle_identity = digest_json({
            "algorithmId": "org.valcorza.swin-classification-finetuner.v1.artifact-bundle",
            "schemaVersion": manifest["schemaVersion"],
            "members": manifest["members"],
            "requiredMemberDigests": sorted(m["digest"] for m in manifest["members"] if m["required"]),
        })
        manifest["bundleDigest"] = bundle_identity
        (stage / "artifact-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        generation_dir = generations / bundle_identity.split(":", 1)[1]
        if generation_dir.exists():
            shutil.rmtree(stage)
        else:
            os.replace(stage, generation_dir)
        (artifact_root / "CURRENT").write_text(generation_dir.name + "\n", encoding="utf-8")
        return manifest, generation_dir
    finally:
        if stage.exists():
            shutil.rmtree(stage)


def train_model(
    dataset_dir: Path,
    handoff_dir: Path,
    staged_weight_path: Path,
    output_dir: Path,
    model_key: str,
    catalog_entry: dict[str, Any],
    epochs: int = 1,
    batch_size: int = 4,
    learning_rate: float = 1e-4,
    weight_decay: float = 0.01,
    seed: int = 20260910,
    device: torch.device = DEVICE,
):
    import torch.nn.functional as functional
    from safetensors.torch import load_file, save_file
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms
    from torchvision.transforms import InterpolationMode

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)

    logical_manifest = json.loads((handoff_dir / "logical-dataset-manifest.json").read_text())
    data_plan = json.loads((handoff_dir / "data-plan.json").read_text())
    semantic_schema = json.loads((handoff_dir / "semantic-dataset-schema.json").read_text())
    validated_manifest = json.loads((handoff_dir / "validated-dataset-manifest.json").read_text())

    label_map = semantic_schema["labelMap"]
    class_names = [label_map[str(i)] for i in range(len(label_map))]
    class_to_target = {name: i for i, name in enumerate(class_names)}
    assignments = {a["sampleId"]: a["split"] for a in data_plan["assignments"]}

    transform = transforms.Compose([
        transforms.Resize((256, 256), interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(NORMALIZATION_MEAN, NORMALIZATION_STD),
    ])

    class ImageDataset(Dataset):
        def __init__(self, sample_ids):
            self.sample_ids = sample_ids

        def __len__(self):
            return len(self.sample_ids)

        def __getitem__(self, index):
            sid = self.sample_ids[index]
            img = load_visual_image(dataset_dir / sid)
            tensor = transform(img)
            target = class_to_target[Path(sid).parts[1]]
            return tensor, target, sid

    train_ids = [s["sampleId"] for s in logical_manifest["samples"] if assignments[s["sampleId"]] == "train"]
    validation_ids = [s["sampleId"] for s in logical_manifest["samples"] if assignments[s["sampleId"]] == "validation"]

    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(ImageDataset(train_ids), batch_size=batch_size, shuffle=True, generator=generator)
    validation_loader = DataLoader(ImageDataset(validation_ids), batch_size=batch_size, shuffle=False)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = timm.create_model(catalog_entry["timmModelName"], pretrained=False, num_classes=len(class_names))
    # Load pretrained backbone weights from the staged and verified safetensors checkpoint
    staged_sd = load_file(staged_weight_path)
    backbone_sd = {k: v for k, v in staged_sd.items() if not k.startswith("head.fc.")}
    model.load_state_dict(backbone_sd, strict=False)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    history = []

    for epoch in range(epochs):
        model.train()
        train_loss_sum, train_count = 0.0, 0
        for inputs, targets, _ids in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = functional.cross_entropy(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * targets.numel()
            train_count += targets.numel()

        model.eval()
        val_loss_sum, val_correct, val_count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets, _ids in validation_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = functional.cross_entropy(outputs, targets)
                val_loss_sum += float(loss.detach().cpu()) * targets.numel()
                val_correct += int((outputs.argmax(dim=1) == targets).sum())
                val_count += targets.numel()

        history.append({
            "epoch": epoch + 1,
            "trainLoss": train_loss_sum / train_count,
            "validationLoss": val_loss_sum / val_count,
            "validationAccuracy": val_correct / val_count,
        })

    transforms_meta = {
        "exifOrientation": EXIF_ORIENTATION,
        "train": [
            {"id": "org.torchvision.resize", "size": [256, 256]},
            {"id": "org.torchvision.to-tensor"},
            {"id": "org.torchvision.normalize", "mean": list(NORMALIZATION_MEAN), "std": list(NORMALIZATION_STD)},
        ],
        "validation": [
            {"id": "org.torchvision.resize", "size": [256, 256]},
            {"id": "org.torchvision.to-tensor"},
            {"id": "org.torchvision.normalize", "mean": list(NORMALIZATION_MEAN), "std": list(NORMALIZATION_STD)},
        ],
        "stochasticAugmentation": False,
    }
    model_config = {
        "schemaVersion": "1.0",
        "modelKey": model_key,
        "timmModelName": catalog_entry["timmModelName"],
        "numClasses": len(class_names),
        "classNames": class_names,
        "input": {"height": 256, "width": 256},
        "transforms": transforms_meta,
    }

    with tempfile.TemporaryDirectory(prefix=".artifact-stage-", dir=output) as stage_dir:
        stage_path = Path(stage_dir)
        weights_path = stage_path / "model.safetensors"
        save_file({k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}, weights_path)
        model_config_path = stage_path / "model-config.json"
        model_config_path.write_text(json.dumps(model_config, indent=2) + "\n", encoding="utf-8")
        model_manifest_path = stage_path / "model_manifest.json"
        model_manifest_path.write_text(json.dumps(build_model_manifest(model_config), indent=2) + "\n", encoding="utf-8")

        artifact_manifest, generation = publish_artifact_bundle(
            output / "artifact",
            (
                ArtifactMemberSource("core.artifact.model.weights", weights_path, "application/vnd.safetensors"),
                ArtifactMemberSource("org.valcorza.swin-classification.model-config", model_config_path, "application/json"),
                ArtifactMemberSource("org.valcorza.timm.model-manifest", model_manifest_path, "application/json"),
            ),
        )

    # Fresh persisted reload verification in-kernel
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    persisted_weights = generation / "model.safetensors"
    fresh_model = timm.create_model(catalog_entry["timmModelName"], pretrained=False, num_classes=len(class_names)).to(device)
    fresh_model.load_state_dict(load_file(persisted_weights, device=str(device)), strict=True)
    fresh_model.eval()

    eval_loss_sum, eval_correct, eval_count = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets, _ids in validation_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = fresh_model(inputs)
            loss = functional.cross_entropy(outputs, targets)
            eval_loss_sum += float(loss.detach().cpu()) * targets.numel()
            eval_correct += int((outputs.argmax(dim=1) == targets).sum())
            eval_count += targets.numel()

    eval_accuracy = eval_correct / eval_count
    eval_cross_entropy = eval_loss_sum / eval_count

    evaluation_report = {
        "schemaVersion": "1.0",
        "artifactManifestDigest": digest_json(artifact_manifest),
        "metrics": [
            {"id": "core.metric.classification.accuracy", "version": "1", "value": eval_accuracy},
            {"id": "org.valcorza.metric.classification.cross-entropy", "version": "1", "value": eval_cross_entropy},
        ],
    }
    (output / "evaluation-report.json").write_text(json.dumps(evaluation_report, indent=2) + "\n", encoding="utf-8")

    run_manifest = {
        "schemaVersion": "1.0",
        "outcome": "SUCCEEDED",
        "observed": {
            "artifactGeneration": generation.name,
            "artifactBundleDigest": artifact_manifest["bundleDigest"],
            "splitCounts": {"train": len(train_ids), "validation": len(validation_ids)},
            "classNames": class_names,
            "history": history,
            "device": str(device),
            "timmVersion": timm.__version__,
            "torchVersion": torch.__version__,
        },
        "reproducibility": "REEXECUTABLE",
    }
    (output / "run-manifest.json").write_text(json.dumps(run_manifest, indent=2) + "\n", encoding="utf-8")
    (output / "result.json").write_text(json.dumps({"schemaVersion": "1.0", "state": "SUCCEEDED"}, indent=2) + "\n", encoding="utf-8")

    return {
        "state": "SUCCEEDED",
        "generation": generation.name,
        "artifactBundleDigest": artifact_manifest["bundleDigest"],
        "metrics": evaluation_report["metrics"],
        "runManifest": run_manifest,
        "evaluationReport": evaluation_report,
    }


# Execute 100% in-kernel fine-tuning
training_result = train_model(
    dataset_dir=DATASET_DIR,
    handoff_dir=HANDOFF_DIR,
    staged_weight_path=staged_file,
    output_dir=TRAINING_DIR,
    model_key=MODEL_KEY,
    catalog_entry=catalog_entry,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    device=DEVICE,
)

run_manifest = training_result["runManifest"]
evaluation_report = training_result["evaluationReport"]
reported_metrics = {metric["id"]: metric["value"] for metric in evaluation_report["metrics"]}

print(json.dumps({
    "state": training_result["state"],
    "device": run_manifest["observed"]["device"],
    "splitCounts": run_manifest["observed"]["splitCounts"],
    "history": run_manifest["observed"]["history"],
    "reportedMetrics": reported_metrics,
    "artifactBundleDigest": run_manifest["observed"]["artifactBundleDigest"],
    "reproducibility": run_manifest["reproducibility"],
}, indent=2))


## 7. Read the evaluation against a trivial baseline

Two metrics are reported by the worker, both on the frozen `validation` split as a **single holdout** — no cross-validation, no repeated runs, therefore no dispersion estimate:

- `core.metric.classification.accuracy` — fraction of validation images whose `argmax` class equals the label. Easy to read, blind to *how* confident the wrong answers were.
- `org.valcorza.metric.classification.cross-entropy` — mean negative log-likelihood of the true class. Sensitive to the score assigned to the correct class, so it moves before accuracy does on tiny datasets.

The **majority-class baseline** is the accuracy you would get by always predicting the most frequent validation class. It is computed from the validator's data plan, so it is correct for BYOD datasets too. On the default balanced sample it is 0.5.

On the synthetic sample these are **tutorial metrics**: the two classes differ in colour and shape, so any working pipeline reaches 1.0 quickly. Treat them as evidence that the pipeline works, not that the model is good.


In [ ]:
validation_sample_ids = sorted(a["sampleId"] for a in data_plan["assignments"] if a["split"] == "validation")
validation_labels = [Path(sample_id).parts[1] for sample_id in validation_sample_ids]  # class = 2nd path component
label_counts = {name: validation_labels.count(name) for name in class_names}
majority_class = max(label_counts, key=label_counts.get)
majority_baseline_accuracy = label_counts[majority_class] / len(validation_labels)

comparison = {
    "estimationProcedure": "single frozen validation holdout (validator-assigned); no dispersion estimate",
    "validationSamples": len(validation_labels),
    "validationLabelCounts": label_counts,
    "majorityClass": majority_class,
    "majorityBaselineAccuracy": majority_baseline_accuracy,
    "modelAccuracy": reported_metrics["core.metric.classification.accuracy"],
    "modelCrossEntropy": reported_metrics["org.valcorza.metric.classification.cross-entropy"],
    "evidenceClass": "tutorial/sanity metrics" if not USE_BYOD_DATASET else "single-holdout metrics on user data",
}
print(json.dumps(comparison, indent=2))
if comparison["modelAccuracy"] < majority_baseline_accuracy:
    print("WARNING: the fine-tuned model does not beat the majority baseline on this holdout. More epochs or more data are needed before drawing any conclusion.")


## 8. Fresh-boundary verification of the exported artifact

A working in-memory model proves nothing about the bytes on disk. This section reproduces what a downstream consumer would do with the artifact and nothing else:

1. copy the published generation directory to a **fresh location** (`WORK/fresh-reload/`) so no path from training is reused;
2. read `artifact-manifest.json` and re-verify every member's SHA-256 (a missing or altered file fails here);
3. rebuild the network from `model-config.json` (`timmModelName`, `numClasses`, `classNames`) and load `model.safetensors` with `strict=True`;
4. run the frozen validation split through it using the finetuner's **own** image decoding (`load_visual_image`, which applies EXIF orientation) and the preprocessing the artifact records (`model-config.json → transforms.validation`: resize size, normalisation mean/std), cross-checked against the worker's `NORMALIZATION_MEAN`/`NORMALIZATION_STD` constants;
5. compare with the worker's evaluation report: accuracy must match **exactly** (deterministic `argmax` on identical inputs), cross-entropy within `1e-4`.

Passing step 3 means *"artifact reloaded"*; passing step 5 means *"artifact reproduces the reported outputs"* — the stronger claim this section exists to make. A per-sample prediction table is kept for export.

> **Known pipeline gap (recorded, not hidden).** The finetuner release exposes no public *load-and-predict* operation; its fresh-reload path lives inside `swin-classification-train`. This section therefore mirrors that path from the artifact contract. Two details the artifact does not record — bicubic interpolation with antialiasing for the resize — are taken from the worker source at the pinned revision (`trainer.py`, the validation transform). Until the worker ships a reconstruction API and records interpolation in `model-config.json`, this is the documented reconstruction recipe, and the equivalence check in step 5 is what proves it matches.

The artifact contains only weights and configuration — no training images — but it was **derived from** your dataset; apply your data's licence and confidentiality rules to it.


In [ ]:
import torch.nn.functional as F
from safetensors.torch import load_file
from torchvision import transforms
from torchvision.transforms import InterpolationMode

# 1. Copy the published generation to a fresh directory.
current_generation = (TRAINING_DIR / "artifact/CURRENT").read_text().strip()
FRESH_DIR = WORK / "fresh-reload"
if FRESH_DIR.exists():
    shutil.rmtree(FRESH_DIR)
shutil.copytree(TRAINING_DIR / "artifact/generations" / current_generation, FRESH_DIR)

# 2. Re-verify every manifest member from the copy.
artifact_manifest = json.loads((FRESH_DIR / "artifact-manifest.json").read_text())


def member_path_of(member) -> str:
    """Each manifest member names its file through a `member-path` relationship."""
    return next(r["path"] for r in member["relationships"] if r["type"] == "org.valcorza.bundle.member-path")


verified_members = []
for member in artifact_manifest["members"]:
    member_file = FRESH_DIR / member_path_of(member)
    if not member_file.is_file():
        raise RuntimeError(f"artifact member missing after copy: {member_file.name} ({member['role']})")
    if sha256_file(member_file) != member["digest"]:
        raise RuntimeError(f"artifact member digest mismatch: {member_file.name} ({member['role']})")
    verified_members.append({"role": member["role"], "file": member_file.name, "digest": member["digest"]})
print("artifact members verified:", json.dumps(verified_members, indent=2))

# 3. Rebuild the model from the artifact contract alone.
model_config = json.loads((FRESH_DIR / "model-config.json").read_text())
assert model_config["classNames"] == class_names, "artifact class order differs from the validator label map"
DEVICE = torch.device("cuda:0")
reloaded_model = timm.create_model(model_config["timmModelName"], pretrained=False, num_classes=model_config["numClasses"])
reloaded_model.load_state_dict(load_file(FRESH_DIR / "model.safetensors"), strict=True)
reloaded_model = reloaded_model.to(DEVICE).eval()

# 4. Score the frozen validation split with the preprocessing the artifact records.
recorded = {step["id"]: step for step in model_config["transforms"]["validation"]}
resize_size = tuple(recorded["org.torchvision.resize"]["size"])
normalize = recorded["org.torchvision.normalize"]
assert resize_size == (model_config["input"]["height"], model_config["input"]["width"]), "artifact resize/input size disagree"
assert tuple(normalize["mean"]) == tuple(NORMALIZATION_MEAN) and tuple(normalize["std"]) == tuple(NORMALIZATION_STD), \
    "artifact normalisation differs from normalisation constants"
assert model_config["transforms"]["exifOrientation"] == "transpose-to-visual-orientation", model_config["transforms"]
preprocess = transforms.Compose([
    transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(normalize["mean"], normalize["std"]),
])


def classify(image_paths):
    """Return (predicted index, softmax scores) per image using the reloaded model and argmax rule."""
    batch = torch.stack([preprocess(load_visual_image(Path(p))) for p in image_paths]).to(DEVICE)
    with torch.no_grad():
        logits = reloaded_model(batch)
    return logits.argmax(dim=1).cpu(), F.softmax(logits, dim=1).cpu(), logits.cpu()


per_sample = []
correct, loss_sum = 0, 0.0
for start in range(0, len(validation_sample_ids), 16):
    ids = validation_sample_ids[start:start + 16]
    predicted, scores, logits = classify([DATASET_DIR / sample_id for sample_id in ids])
    targets = torch.tensor([class_names.index(Path(sample_id).parts[1]) for sample_id in ids])
    loss_sum += float(F.cross_entropy(logits, targets, reduction="sum"))
    correct += int((predicted == targets).sum())
    for i, sample_id in enumerate(ids):
        per_sample.append({"sampleId": sample_id, "label": class_names[int(targets[i])], "predicted": class_names[int(predicted[i])],
                           **{f"score_{name}": float(scores[i][k]) for k, name in enumerate(class_names)}})

# 5. Compare with the reported metrics.
reload_check = {
    "reloadedAccuracy": correct / len(validation_sample_ids),
    "reportedAccuracy": reported_metrics["core.metric.classification.accuracy"],
    "reloadedCrossEntropy": loss_sum / len(validation_sample_ids),
    "reportedCrossEntropy": reported_metrics["org.valcorza.metric.classification.cross-entropy"],
    "crossEntropyTolerance": 1e-4,
}
reload_check["accuracyMatches"] = reload_check["reloadedAccuracy"] == reload_check["reportedAccuracy"]
reload_check["crossEntropyMatches"] = abs(reload_check["reloadedCrossEntropy"] - reload_check["reportedCrossEntropy"]) <= reload_check["crossEntropyTolerance"]
print(json.dumps(reload_check, indent=2))
if not (reload_check["accuracyMatches"] and reload_check["crossEntropyMatches"]):
    raise RuntimeError("The reloaded artifact does not reproduce the reported metrics. Do not ship this artifact.")
print(f"Fresh-boundary verification PASSED: artifact reproduces the reported metrics on {len(per_sample)} validation samples.")


## 9. Classify a new image

Real use means images the model has never seen. The default is a freshly drawn `warm`-style image that is in neither split; set `USE_BYOD_IMAGE = True` to upload one of your own instead (any of the accepted extensions; EXIF orientation is honoured exactly as during training).

The decision rule is `argmax(logits)`. The softmax scores are **uncalibrated class scores** — larger means the model favours that class, but 0.9 does not mean "90 % chance of being right". The pipeline ships no confidence threshold; if a downstream system needs one, calibrate it on labelled data from your deployment domain.

**Look for:** `predictedClass` and the per-class scores, which sum to 1.


In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ""  # @param {type:"string"}

NEW_IMAGE_DIR = WORK / "new-images"
NEW_IMAGE_DIR.mkdir(exist_ok=True)
if USE_BYOD_IMAGE:
    if BYOD_IMAGE_PATH:
        new_image_path = Path(BYOD_IMAGE_PATH)
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise RuntimeError("Not running in Colab: set BYOD_IMAGE_PATH to an image already present in this runtime.") from error
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Upload exactly one image.")
        new_image_path = NEW_IMAGE_DIR / next(iter(uploaded))
        new_image_path.write_bytes(next(iter(uploaded.values())))
    if new_image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        raise RuntimeError(f"unsupported image extension {new_image_path.suffix!r}; accepted: {sorted(IMAGE_EXTENSIONS)}")
    new_image_origin = "user-supplied image (BYOD)"
else:
    new_image_path = NEW_IMAGE_DIR / "synthetic-new-warm.png"
    image = Image.new("RGB", (256, 256), (190, 65, 35))
    ImageDraw.Draw(image).ellipse((68, 68, 188, 188), outline=(255, 240, 220), width=12)
    image.save(new_image_path)
    new_image_origin = "synthetic new image drawn by this notebook (not in train/val)"

predicted, scores, _ = classify([new_image_path])
prediction = {
    "input": new_image_path.name,
    "inputOrigin": new_image_origin,
    "inputSha256": sha256_file(new_image_path),
    "decisionRule": "argmax(logits)",
    "predictedClass": class_names[int(predicted[0])],
    "uncalibratedSoftmaxScores": {name: float(scores[0][k]) for k, name in enumerate(class_names)},
    "classOrder": class_names,
}
print(json.dumps(prediction, indent=2))


## 10. Export machine-readable results and provenance

Four files are written to `WORK/exports/` (in Colab, open the folder icon on the left to download them):

| File | Contents |
|---|---|
| `validation-predictions.csv` | one row per validation sample: `sampleId`, `label`, `predicted`, one `score_<class>` column per class in `classOrder` |
| `metrics.json` | worker-reported metrics, majority baseline, and the fresh-boundary reload comparison |
| `prediction.json` | the new-image prediction from Section 9 |
| `provenance.json` | pipeline/worker/model revisions and digests, runtime, training configuration, dataset identity (`logicalDatasetDigest`), artifact bundle digest |

The deployable artifact itself is the generation directory under `training-output/artifact/generations/<generation>/` (`model.safetensors`, `model-config.json`, `model_manifest.json`, `artifact-manifest.json`); `model_manifest.json` is what DIMER's timm inference service consumes to reconstruct the classifier. No secrets or tokens are written to any export.


In [ ]:
EXPORT_DIR = WORK / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

with (EXPORT_DIR / "validation-predictions.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(per_sample[0].keys()))
    writer.writeheader()
    writer.writerows(per_sample)

metrics_export = {
    "notebookProfile": "E2E",
    "notebookSpec": "1.0",
    "reportedMetrics": reported_metrics,
    "comparison": comparison,
    "freshBoundaryReload": reload_check,
}
provenance_export = {
    "notebookProfile": "E2E",
    "notebookSpec": "1.0",
    "pipeline": {"repository": "kurtvalcorza/swin-classification-pipeline", "revision": PIPELINE_REF, "pipelineId": pipeline_manifest["pipelineId"], "taskProfile": pipeline_manifest["taskProfile"]},
    "workers": {
        "validator": {"revision": VALIDATOR_REF, "workerReleaseDigest": pipeline_manifest["validatorWorkerReleaseDigest"]},
        "finetuner": {"revision": FINETUNER_REF, "workerReleaseDigest": pipeline_manifest["finetunerWorkerReleaseDigest"]},
    },
    "baseModel": model_identity,
    "runtime": runtime,
    "dataset": {**dataset_origin, "logicalDatasetDigest": logical_manifest["logicalDatasetDigest"], "classNames": class_names, "splitCounts": split_counts},
    "training": {"method": "core.training.supervised-finetuning (full network, AdamW, cross-entropy)", "epochs": EPOCHS, "batchSize": BATCH_SIZE, "learningRate": LEARNING_RATE, "seed": SEED,
                 "reproducibility": run_manifest["reproducibility"], "device": run_manifest["observed"]["device"]},
    "artifact": {"generation": current_generation, "bundleDigest": artifact_manifest["bundleDigest"], "members": verified_members, "format": "safetensors + model-config.json + artifact-manifest.json"},
    "evaluation": comparison,
    "freshBoundaryReload": reload_check,
}
(EXPORT_DIR / "metrics.json").write_text(json.dumps(metrics_export, indent=2) + "\n")
(EXPORT_DIR / "prediction.json").write_text(json.dumps(prediction, indent=2) + "\n")
(EXPORT_DIR / "provenance.json").write_text(json.dumps(provenance_export, indent=2) + "\n")
for path in sorted(EXPORT_DIR.iterdir()):
    print(f"{path.stat().st_size:>8} bytes  {path}")


## Interpretation and limits

A successful top-to-bottom run **proves**, for the runtime printed in Section 1:

- the pipeline repository was checked out at the immutable revision the release pins;
- the checkpoint bytes matched the allowlisted SHA-256 before any model was built;
- the in-notebook validator accepted the dataset and froze its identity;
- in-kernel gradient fine-tuning completed on `cuda:0`, published a content-addressed artifact, and evaluated the *persisted* model;
- that artifact, copied to a fresh directory and rebuilt from its own contract, reproduced reported accuracy exactly and cross-entropy within tolerance;
- a new image could be classified from the reloaded artifact and every result was exported with provenance.

It does **not** prove: ImageNet-level or real-domain accuracy (the default data is synthetic and tiny; even BYOD metrics are a single holdout with no dispersion estimate), calibration of the softmax scores, robustness, fairness, safety, or production fitness. Formal accelerator qualification remains limited to the finetuner's recorded qualification packet; a clean run on Colab or Kaggle is execution evidence for *that* runtime only. Static notebook checks are not execution evidence — see `tutorials/RELEASE_VERIFICATION.md` for the recorded clean-runtime run of this notebook revision.

## Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `RuntimeError: Anonymous clone of the pipeline repository was refused…` | The repository is public, so this means a private fork/mirror or no network access to GitHub. Check connectivity first; for a private fork add a `GITHUB_TOKEN` secret (Colab: key icon → Secrets, enable notebook access; Kaggle: Add-ons → Secrets) and rerun Section 1. |
| `git clone` fails with `403`/`Authentication failed` after the token fallback | The token exists but lacks read access to the (forked/mirrored) pipeline repository. |
| `CUDA is required…` or `ACCELERATOR_UNAVAILABLE` / `ACCELERATOR_MISMATCH` | Runtime has no `cuda:0`. Colab: Runtime → Change runtime type → GPU (T4), then Run all. |
| `timm … is active but 1.0.28 was installed` / Pillow WARNING in Section 1 | A package was imported before the pinned install (re-run in the same kernel, or a pre-warmed runtime). Runtime → Restart session, then run from the top. |
| `VISION_MISSING_VALIDATION_SPLIT` / `VISION_AMBIGUOUS_VALIDATION_SPLIT` | BYOD ZIP lacks `val/` (or has both `val/` and `valid/`). Fix the folder layout and rerun Sections 3–4. |
| `VISION_UNRECOGNIZED_IMAGE_EXTENSION` or a decode error | A non-image or corrupt file is inside a class folder. Remove it; nothing is skipped silently by design. |
| `VISION_UNKNOWN_ROOT_ENTRY`, `VISION_UNASSIGNED_SPLIT_FILE`, `VISION_NESTED_CLASS_DIRECTORY` | The ZIP carried extra files (`README.txt`, `.DS_Store`, `__MACOSX/`) or nested folders. Only `train|val|valid|test/<class>/<image>` is accepted; clean the folder and re-zip. |
| `VISION_CLASS_NAME_COLLISION` | Two class folders differ only by case or Unicode form. Rename one. |
| `CUDA out of memory` during Section 6 | Lower `BATCH_SIZE` (or choose the Tiny model) and rerun Section 6. |
| Checkpoint digest mismatch | Bytes served by the Hub differ from the catalog pin. Do not proceed; report it against the pipeline repository. |

## Next experiments

- Turn on `USE_BYOD_DATASET` with a small real image folder and compare the majority baseline with the model — this is the first result that says anything about your domain.
- Raise `EPOCHS` and watch `history`: on the synthetic sample validation loss collapses within an epoch; on real data look for the point where validation loss stops improving.
- Switch `MODEL_KEY` to the Small variant and compare cross-entropy at equal epochs and seed.
- Run twice with the same `SEED` and diff `provenance.json`: identical accuracy, slightly different cross-entropy and artifact digest — that is what *re-executable, not bitwise deterministic* means.
- Hand `fresh-reload/` to a separate process or machine and repeat Section 8 there; that is the boundary DIMER serving relies on.
